In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
import os
from row_generate_func import *
from auto_insert_func import *

from sqlalchemy.orm import Session

engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [3]:
# --- Main workflow ---
output_folder = '/workspaces/crmprtd/fern/v4_after_metadata_update/output/'
file_name = 'Fern_raw_db_station_matched.csv'

df_match = pd.read_csv(output_folder + file_name)

df_match

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
0,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Air Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,TempC,celsius,NaN,NaN,NaN,NaN,batch4
1,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Air Temp 2,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5
2,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2
3,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 25 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_25cm,celsius,NaN,NaN,NaN,NaN,batch4
4,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 50 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_50cm,celsius,NaN,NaN,NaN,NaN,batch4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2024-10-24 09:00:00,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1
423,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2024-10-24 09:00:00,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1
424,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2024-10-24 09:00:00,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,373.0,batch1
425,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2024-10-24 09:00:00,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1


In [4]:
station_names = df_match['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id, m.lat, m.lon, m.elev
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    db_stations = pd.read_sql(query, conn, params={"station_names": station_names})

In [5]:
merged = df_match.merge(
    db_stations,
    on='station_name',
    how='left'
)

match_info = merged.rename(
    columns={
        'native_id': 'native_id_db',
        'lat_y': 'lat_db',
        'lon_y': 'lon_db',
        'elev_y': 'elev_db'
    }
)

match_info = match_info.drop(columns=['native_id_raw', 'lat_x', 'lon_x', 'elev_x'])

match_info

,station_name,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,native_id_db,lat_db,lon_db,elev_db
0,Atlin School,Atlin school,Air Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,TempC,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
1,Atlin School,Atlin school,Air Temp 2,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5,AtlSch,59.574000,-133.687000,705.0
2,Atlin School,Atlin school,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
3,Atlin School,Atlin school,Gd Temp 25 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_25cm,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
4,Atlin School,Atlin school,Gd Temp 50 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_50cm,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,Willow-BowronWx,WillowBowron PGTIS 2,Solar Radiation,W/m²,2007-07-31 15:00:00,2024-10-24 09:00:00,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0
423,Willow-BowronWx,WillowBowron PGTIS 2,Temp,°C,2007-07-31 15:00:00,2024-10-24 09:00:00,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0
424,Willow-BowronWx,WillowBowron PGTIS 2,Wetness,%,2008-06-12 13:00:00,2024-10-24 09:00:00,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0
425,Willow-BowronWx,WillowBowron PGTIS 2,Wind Direction,ø,2007-07-31 15:00:00,2024-10-24 09:00:00,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0


### Batch 1, for the unique variables (only one sensor)

In [6]:
match_info_uniq_var = match_info[match_info['db_var_match'].notna()]
match_info_uniq_var

,station_name,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,native_id_db,lat_db,lon_db,elev_db
0,Atlin School,Atlin school,Air Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,TempC,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
2,Atlin School,Atlin school,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
3,Atlin School,Atlin school,Gd Temp 25 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_25cm,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
4,Atlin School,Atlin school,Gd Temp 50 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_50cm,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
5,Atlin School,Atlin school,Gd Temp 75 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_75cm,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,Willow-BowronWx,WillowBowron PGTIS 2,Solar Radiation,W/m²,2007-07-31 15:00:00,2024-10-24 09:00:00,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0
423,Willow-BowronWx,WillowBowron PGTIS 2,Temp,°C,2007-07-31 15:00:00,2024-10-24 09:00:00,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0
424,Willow-BowronWx,WillowBowron PGTIS 2,Wetness,%,2008-06-12 13:00:00,2024-10-24 09:00:00,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0
425,Willow-BowronWx,WillowBowron PGTIS 2,Wind Direction,ø,2007-07-31 15:00:00,2024-10-24 09:00:00,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1,PGTIS2,53.772577,-122.724634,596.0


In [7]:

data_path = '/fern_data/FERNNorth2024_VF/WxData24'

all_rows = []

for i, row in match_info_uniq_var.iterrows():
    station = row["station_name"]
    variable = row["variable"]

    print(f"\n🔍 [{i+1}/{len(match_info_uniq_var)}] Processing {station} — {variable}")

    new_rows = extract_new_data_rows(row, data_path)
    all_rows.extend(new_rows)

    if len(new_rows) > 0:
        print(f"  ✅ {len(new_rows)} new records found outside DB time range.")
    else:
        print("  ⚠️ No new records found (matches DB time range exactly).")


🔍 [1/401] Processing Atlin School — Air Temp
  ✅ 119237 new records found outside DB time range.

🔍 [3/401] Processing Atlin School — DewPt
  ✅ 111039 new records found outside DB time range.

🔍 [4/401] Processing Atlin School — Gd Temp 25 cm
  ✅ 119237 new records found outside DB time range.

🔍 [5/401] Processing Atlin School — Gd Temp 50 cm
  ✅ 119237 new records found outside DB time range.

🔍 [6/401] Processing Atlin School — Gd Temp 75 cm
  ✅ 119237 new records found outside DB time range.

🔍 [7/401] Processing Atlin School — Gust Speed
  ✅ 119237 new records found outside DB time range.

🔍 [8/401] Processing Atlin School — RH
  ✅ 111039 new records found outside DB time range.

🔍 [9/401] Processing Atlin School — Rain
  ✅ 119231 new records found outside DB time range.

🔍 [10/401] Processing Atlin School — Sfc Temp
  ✅ 119237 new records found outside DB time range.

🔍 [11/401] Processing Atlin School — Solar Radiation
  ✅ 119237 new records found outside DB time range.

🔍 [12/

### Insert

In [13]:
len(all_rows)

6778465

In [12]:
db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"

safe_insert_rows(
    all_rows,
    log_file_path=output_folder + 'fern_all_station_insert.log',
    db_url=db_url,
    chunk_size=5000,        # SAFE starting point
    bulk_chunk_size=1000,   # internal DB bulk insert size
)

🚀 Starting insert: 6778465 rows in 1356 chunks
➡️  Processing rows 0–4999
{"asctime": "2025-12-17 19:28:32,510", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2025-12-17 19:28:33,146", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 900 inserted, 100 skipped, 0 failed"}
{"asctime": "2025-12-17 19:28:33,834", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 1900 inserted, 100 skipped, 0 failed"}
{"asctime": "2025-12-17 19:28:34,603", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 2900 inserted, 100 skipped, 0 failed"}
{"asctime": "2025-12-17 19:28:35,324", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 3900 inserted, 100 skipped, 0 failed"}
{"asctime": "2025-12-17 19:28:36,050", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 4900 inserted, 100 skipped, 0 failed"}
{"asc